# Debugging Workflow



```{contents}
:local:
:depth: 2
```


```{index} Turn a Symptom into a Reproducible Question
```

## Turn a Symptom into a Reproducible Question



A report says that three expense entries, 12, 8 and 5, total 20. Saying that the total is wrong identifies a symptom, but it does not yet locate the defect. First derive the expected total independently: 12 + 8 + 5 is 25. Keep the three inputs fixed. A changing input set makes it difficult to tell whether a code change repaired the problem or merely avoided it.

Suppose the program loops while index is less than the number of entries minus one. The loop body may contain correct addition and still omit the final item. This is why debugging begins with the contract and a concrete failing case rather than rewriting whichever line looks suspicious. At this point, students can use a simple three-value loop; the same workflow later applies to arrays and object state.

For related explanation, see the further reading below.[^development-reading]


```{index} The Workflow
```



```{index} The Workflow
```

## The Workflow



The steps below develop this workflow. Form and challenge hypotheses while inspecting state; use typical bug patterns as candidates for a small repair, then retain the regression evidence. The complete worked investigation follows the reference steps.



Use this sequence every time:

1. Reproduce reliably
2. Minimize the failing case
3. Observe state with tools
4. Form a specific hypothesis
5. Patch one thing
6. Re-test and check for regressions

```{index} Step 1: Reproduce Reliably
```



```{index} Step 1: Reproduce Reliably
```

### Step 1: Reproduce Reliably




Capture:
- exact input
- exact command
- expected behavior
- actual behavior

Without reproducibility, debugging quality collapses.

```{index} Step 2: Minimize the Failing Case
```



```{index} Step 2: Minimize the Failing Case
```

### Step 2: Minimize the Failing Case




Start with a *buggy* example — an off-by-one error in a sum loop:

In [ ]:
// Buggy: skips the last element
int[] scores = { 85, 92, 78, 90, 88 };
int total = 0;

for (int i = 0; i < scores.Length - 1; i++)  // BUG: should be <= or just <
    total += scores[i];

Console.WriteLine($"Total: {total}");    // prints 345 instead of 433
Console.WriteLine($"Count: {scores.Length}");

Total: 345
Count: 5


Fix: change the loop condition to `i < scores.Length`:

In [ ]:
int[] scores = { 85, 92, 78, 90, 88 };
int total = 0;

for (int i = 0; i < scores.Length; i++)
    total += scores[i];

Console.WriteLine($"Total: {total}");   // 433
Console.WriteLine($"Average: {total / (double)scores.Length:F1}");  // 86.6

Total: 433
Average: 86.6


```{index} Step 3: Breakpoints and Stepping in VS Code
```



```{index} Step 3: Breakpoints and Stepping in VS Code
```

### Step 3: Breakpoints and Stepping in VS Code




- Set a breakpoint **before** the suspected line
- **Step Over** (`F10`) — execute one line, staying in current method
- **Step Into** (`F11`) — follow execution into a called method
- **Step Out** (`Shift+F11`) — exit the current method

Set the breakpoint on the `total += scores[i]` line and watch `total` after each iteration.

```{index} Step 4: Watch and Call Stack
```



```{index} Step 4: Watch and Call Stack
```

### Step 4: Watch and Call Stack




Inspect:
- key variables before and after transformations
- the call stack to confirm control flow assumptions
- branch conditions at decision points (`if`, loop bounds)

Add a **Watch** expression for `scores[i]` and `total` to see each step.

```{index} Test a Hypothesis Against Execution
```



```{index} Test a Hypothesis Against Execution
```

#### Test a Hypothesis Against Execution



State a hypothesis before stepping: perhaps the loop visits only two entries. Put a breakpoint before the loop, not after the incorrect report has already been produced. Observe index, the current input, and the accumulated total. Step over the update, then watch the loop condition. In VS Code, a breakpoint pauses execution when the running program reaches that source location; it does not itself execute or repair the line. Step Into follows a call, while Step Over completes that call without opening its internal steps.

| Observation point | Index being processed | Total after the update |
| --- | --- | --- |
| First iteration | 0 | 12 |
| Second iteration | 1 | 20 |
| Expected third iteration | 2 | 25 |

If execution never reaches the third row, the evidence supports an iteration-boundary defect. If all three updates occur but the final result is still 20, investigate later reassignment or output instead. A useful observation distinguishes competing explanations. Repeatedly pressing Step without a question can produce many observations without improving understanding.

The call stack tells you how execution reached the paused method. Local variables belong to the current call; selecting another frame can show another call's values. A watch expression is a view of a value, not proof that the value satisfies a business rule. Prefer observations that do not alter program state. A debugging expression that invokes a mutating method can itself change the behavior you are trying to understand.




```{index} Step 5: Typical Bug Patterns
```



```{index} Step 5: Typical Bug Patterns
```

### Step 5: Typical Bug Patterns




| Pattern | Example |
|---------|--------|
| Off-by-one | `i < Length - 1` instead of `i < Length` |
| Stale loop state | reusing a variable without resetting it |
| Null/empty assumption | skipping `IsNullOrWhiteSpace` check |
| Parsing without validation | `int.Parse` on unvalidated input |

```{index} Step 6: Validate the Fix
```



```{index} Step 6: Validate the Fix
```

### Step 6: Validate the Fix




After patching:
1. Re-run the **original** failing input
2. Run edge cases (empty array, single element, max values)
3. Run unit tests if they exist

In [ ]:
// Edge case: single element
int[] single = { 42 };
int t = 0;
for (int i = 0; i < single.Length; i++) t += single[i];
Console.WriteLine(t);  // 42

// Edge case: empty array
int[] empty = {};
int t2 = 0;
for (int i = 0; i < empty.Length; i++) t2 += empty[i];
Console.WriteLine(t2);  // 0

42
0


```{index} Keep Evidence After the Repair
```



```{index} Keep Evidence After the Repair
```

#### Keep Evidence After the Repair



After changing the boundary, rerun the exact 12/8/5 case and record expected 25 versus actual 25. Then check no entries, one entry, and a final entry whose omission would be obvious. These checks do not prove every possible input correct, but they defend against the diagnosed class of defect. Keep the failing case as a regression check rather than replacing it with a new convenient example.

Guided variation: the original failing report contains only one expense of 7 and prints zero. Predict what this says about the loop boundary. Then consider a report that prints 14 instead. The first observation suggests a skipped iteration; the second suggests duplicated processing or a different arithmetic/state problem. The workflow should follow the evidence rather than treating every wrong total as the same bug.


Debugging is a repeatable process, not guesswork.

```{index} Turn a Symptom into a Reproducible Question
```



```{index} Debug One Familiar Loop from Symptom to Evidence
```

## Debug One Familiar Loop from Symptom to Evidence


Before the retained array and CSV applications, use a calculation that needs only methods, decisions, and loops. A shop awards one credit for each of the first `days` days. With three days, the expected credits are 1 + 2 + 3 = 6. This expectation comes from the problem, not from running the implementation.

The following complete cell deliberately contains a logic defect. It compiles, but the loop stops too soon. A compiler cannot infer that the business rule requires the last day.


In [1]:
using System;
static int Credits(int days)
{
    int total = 0;
    for (int day = 1; day < days; day++) // Intentional defect: excludes the last day.
        total += day;
    return total;
}
Console.WriteLine($"Expected 6; actual {Credits(3)}");


Expected 6; actual 3


Save this code as the Program.cs of a console project using the Appendix B workflow. Run `dotnet run` to reproduce the failure before starting a debugger. In VS Code, open the folder containing the project, set a breakpoint on `total += day`, and start the C# debugger for that project. The browser cell can demonstrate the result; the editor debugger supplies paused state.

Write the hypothesis: “the loop omits day three.” A breakpoint pauses **before** the marked statement executes. At its first stop, watch `day`, `days`, and `total`. Step Over the update, then continue to the next stop. Use the method's own frame in the Call Stack panel to inspect its local values. The call stack explains who called the method; it is not a list of all previous loop iterations.

| Observation | day | total before addition | total after addition |
| --- | ---: | ---: | ---: |
| First stopped iteration | 1 | 0 | 1 |
| Second stopped iteration | 2 | 1 | 3 |
| Third iteration required by the contract | 3 | 3 | 6 |

The third row does not occur in the defective run. After the second addition, `day` becomes three and `day < days` is false. This distinguishes a missing iteration from an incorrect addition. If all three additions had occurred, investigate a later overwrite or the printed expression instead of changing the boundary blindly.

Repair only the condition to `day <= days`. Step through the same input again: a stop with day three should now appear. Keep the old input; changing both the data and the program would weaken the evidence.


In [1]:
using System;
static int Credits(int days)
{
    if (days < 0) throw new ArgumentOutOfRangeException(nameof(days));
    int total = 0;
    for (int day = 1; day <= days; day++) total += day;
    return total;
}
static void Check(int days, int expected)
{
    int actual = Credits(days);
    Console.WriteLine($"days={days}: expected={expected}, actual={actual}, pass={actual == expected}");
}
Check(0, 0);
Check(1, 1);
Check(3, 6);
Check(4, 10);


days=0: expected=0, actual=0, pass=True
days=1: expected=1, actual=1, pass=True
days=3: expected=6, actual=6, pass=True
days=4: expected=10, actual=10, pass=True


```{index} Use a Different Symptom to Challenge the Hypothesis
```

### Use a Different Symptom to Challenge the Hypothesis


The output compares independently derived expectations with actual results. Zero checks that the loop can run no times; one exposes the original omission immediately; three preserves the original failure; four checks another ordinary case. The negative-input guard is a contract improvement, not the cause of the original wrong total. Record those as separate changes when reviewing a repair.

Guided task: suppose a variant returns 7 for three days. Fill a trace with starting total one instead of zero. Does changing `<` to `<=` alone explain or repair that symptom? Then add an intentional `total = day` in place of `total += day`. Predict the output for four days before debugging.

```{dropdown} Compare the defects
Starting at one adds an unwanted extra credit. With the corrected boundary it yields seven for three days. Assignment instead of accumulation discards earlier credits and returns four for four days. The boundary, initialization and update are separate hypotheses. A trace showing state before and after the update distinguishes them.
```

Use the existing array example after Chapter 7 to transfer this workflow to indexed input. Use the CSV exercise after Chapter 9 to investigate field selection and normalization. Their original tasks remain available; they are later applications rather than assumed knowledge for this first debugger investigation.


```{index} Practice
```

## Practice




Take a method that parses a CSV line:

1. Add one intentional bug (wrong field index, missing `Trim`)
2. Set a breakpoint at the parse start
3. Add a watch on `parts.Length` and each field
4. Step through `TryParse` calls
5. Identify the bug and fix it
6. Re-run with both valid and malformed input

```{rubric} Footnotes
```


[^development-reading]: Further reading: [Microsoft, C# Debugging in VS Code](https://code.visualstudio.com/docs/csharp/debugging). The worked business scenarios and guided reasoning here are original teaching examples.
